# MLB Stats Agent (LangChain + statsapi) - Step-by-step Demo

이 노트북은 다음을 단계별로 검증합니다.

1) 환경/의존성 설정
2) 도구 개별 테스트: `find_player_id`, `get_player_stats`
3) 에이전트 프롬프트(슬롯 필링) 확인: 부족한 정보 되묻기
4) 비교 여부 확인 로직(추가 질문) 확인
5) 멀티턴 대화 흐름 확인

주의: 실제 호출에는 OpenAI API 키가 필요합니다.


In [1]:
# OpenAI API Key 설정 (옵션)
# 아래 두 방법 중 하나를 사용하세요.

# 1) 직접 입력 방식 (콘솔/노트북에서 안전한 입력)
try:
    from getpass import getpass
    _k = getpass("Enter OpenAI API Key (빈칸이면 건너뜀): ")
    if _k.strip():
        import os
        os.environ["OPENAI_API_KEY"] = _k.strip()
        print("OPENAI_API_KEY set via input")
except Exception as _e:
    print("Skip input method:", _e)

# 2) 변수에 직접 대입 (편의용)
# import os
# os.environ["OPENAI_API_KEY"] = "sk-..."



OPENAI_API_KEY set via input


In [2]:
import os
import json
from pprint import pprint

import statsapi

from mlb_multiturn_agent import find_player_id, get_player_stats, build_agent

# 환경 확인
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("statsapi version OK; sample lookup length:", len(statsapi.lookup_player("Harper")))


OPENAI_API_KEY set: True
statsapi version OK; sample lookup length: 2


In [10]:
# 최신 코드 반영을 위해 모듈 리로드
import importlib
import mlb_multiturn_agent as mma
importlib.reload(mma)
from mlb_multiturn_agent import find_player_id, get_player_stats, build_agent
print("Reloaded mlb_multiturn_agent; tools rebound.")


Reloaded mlb_multiturn_agent; tools rebound.


In [4]:
# Tool 1: find_player_id 테스트

print("Query: 'Harper'")
res1 = find_player_id.invoke({"player_name": "Harper"})
print(res1)

print("\nQuery: 'Bryce Harper' (정확 이름)")
res2 = find_player_id.invoke({"player_name": "Bryce Harper"})
print(res2)

print("\nQuery: 'Shohei Ohtani' (일본 표기/영문 표기 다양성 체크)")
res3 = find_player_id.invoke({"player_name": "Shohei Ohtani"})
print(res3)


Query: 'Harper'
[Tool Log] find_player_id 호출됨: Harper
AMBIGUOUS: Multiple players found. Choose one by id. Candidates: Bryce Harper | 1B |  | id=547180 ; Will Warren | P |  | id=701542

Query: 'Bryce Harper' (정확 이름)
[Tool Log] find_player_id 호출됨: Bryce Harper
547180

Query: 'Shohei Ohtani' (일본 표기/영문 표기 다양성 체크)
[Tool Log] find_player_id 호출됨: Shohei Ohtani
660271


In [5]:
# Tool 2: get_player_stats 테스트 (season / career)

# 'Bryce Harper' 가정
harper_id = find_player_id.invoke({"player_name": "Bryce Harper"})
print("Harper ID:", harper_id)

print("\nSeason 2023, hitting:")
res_season = get_player_stats.invoke({
    "player_id": harper_id if isinstance(harper_id, str) else str(harper_id),
    "time_info": "2023",
    "stat_type": "hitting",
})
pprint(res_season if isinstance(res_season, dict) else res_season)

print("\nCareer, hitting:")
res_career = get_player_stats.invoke({
    "player_id": harper_id if isinstance(harper_id, str) else str(harper_id),
    "time_info": "career",
    "stat_type": "hitting",
})
pprint(res_career if isinstance(res_career, dict) else res_career)

print("\nSeason 2021, fielding (가능 시):")
res_fielding = get_player_stats.invoke({
    "player_id": harper_id if isinstance(harper_id, str) else str(harper_id),
    "time_info": "2021",
    "stat_type": "fielding",
})
pprint(res_fielding if isinstance(res_fielding, dict) else res_fielding)


[Tool Log] find_player_id 호출됨: Bryce Harper
Harper ID: 547180

Season 2023, hitting:
[Tool Log] get_player_stats 호출됨: 547180, 2023, hitting
{'group': 'hitting',
 'player_id': '547180',
 'seasons': [{'group': 'hitting',
              'raw': {'active': True,
                      'bat_side': 'Left',
                      'current_team': 'Philadelphia Phillies',
                      'first_name': 'Bryce',
                      'id': 547180,
                      'last_name': 'Harper',
                      'last_played': None,
                      'mlb_debut': '2012-04-28',
                      'nickname': 'Harp',
                      'pitch_hand': 'Right',
                      'position': '1B',
                      'stats': [{'group': 'hitting',
                                 'season': '2023',
                                 'stats': {'age': 30,
                                           'airOuts': 93,
                                           'atBats': 457,
                   

In [6]:
# 에이전트 빌드 및 멀티턴 대화 흐름 테스트

agent = build_agent()

# 1) 정보 부족한 상태에서 질문 → 되묻기 기대
resp1 = agent.invoke({"input": "오타니 성적 알려줘"})
print("[1]", resp1.get("output"))

# 2) 시즌 정보 제공 → 여전히 stat_type 부족 예상 → 재질문
resp2 = agent.invoke({"input": "2023년"})
print("[2]", resp2.get("output"))

# 3) stat_type 제공 → 이제 find_player_id → get_player_stats 호출 예상
resp3 = agent.invoke({"input": "타격"})
print("[3]", resp3.get("output"))

# 4) 비교 여부 질문에 응답 후 확정 분석(케이스에 따라 변동)
resp4 = agent.invoke({"input": "한 선수만 분석해줘"})
print("[4]", resp4.get("output"))


/data1/snuppy/baseball_project/mlb_multiturn_agent.py:240: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(




> Entering new AgentExecutor chain...
오타니 선수의 성적이 궁금하시군요. 어떤 시즌의 기록을 알려드릴까요? 아니면 커리어 전체 성적을 원하시나요?

> Finished chain.
[1] 오타니 선수의 성적이 궁금하시군요. 어떤 시즌의 기록을 알려드릴까요? 아니면 커리어 전체 성적을 원하시나요?


> Entering new AgentExecutor chain...
2023년 오타니 선수의 어떤 기록이 궁금하신가요? 타격 성적, 투구 성적, 또는 수비 성적 중 어떤 것을 원하시나요?

> Finished chain.
[2] 2023년 오타니 선수의 어떤 기록이 궁금하신가요? 타격 성적, 투구 성적, 또는 수비 성적 중 어떤 것을 원하시나요?


> Entering new AgentExecutor chain...

Invoking: `find_player_id` with `{'player_name': 'Shohei Ohtani'}`


[Tool Log] find_player_id 호출됨: Shohei Ohtani
660271
Invoking: `get_player_stats` with `{'player_id': '660271', 'time_info': '2023', 'stat_type': 'hitting'}`


[Tool Log] get_player_stats 호출됨: 660271, 2023, hitting
{'player_id': '660271', 'time': '2023', 'type': 'season', 'group': 'hitting', 'seasons': [{'season': '2023', 'type': 'season', 'group': 'hitting', 'raw': {'id': 660271, 'first_name': 'Shohei', 'last_name': 'Ohtani', 'active': True, 'current_team': 'Los Angeles Dodgers', 'position': 'TWP', 'nic

## 참고
- 이름 모호 시 `find_player_id`는 `AMBIGUOUS:`로 후보를 제시합니다. 에이전트는 이때 사용자가 고르는지 되묻습니다.
- `time_info`는 `career` 또는 `YYYY`에 우선 대응합니다. 날짜 범위(`YYYY-MM-DD_YYYY-MM-DD`)는 TODO입니다.
- `stat_type`은 `hitting`, `pitching`, `fielding`을 지원합니다.
- 비교 분석은 시스템 프롬프트에 따라 사용자 의도를 재확인 후 진행합니다.


In [11]:
# 간단 대화 루프 (노트북용)
agent = build_agent()
print("엔터로 이어서, 'exit'로 종료")

while True:
    q = input("You: ").strip()
    if not q:
        continue
    if q.lower() in {"exit", "quit"}:
        print("Bye!")
        break
    resp = agent.invoke({"input": q})
    print("Agent:", resp.get("output", ""))

엔터로 이어서, 'exit'로 종료


> Entering new AgentExecutor chain...
스쿠발 선수의 성적을 확인해드리겠습니다. 하지만 선수의 이름이 "스쿠발"로만 제공되어 있어, 정확한 영문 이름이 필요합니다. 스쿠발 선수의 공식 영문 이름이 무엇인지 알려주실 수 있나요?

> Finished chain.
Agent: 스쿠발 선수의 성적을 확인해드리겠습니다. 하지만 선수의 이름이 "스쿠발"로만 제공되어 있어, 정확한 영문 이름이 필요합니다. 스쿠발 선수의 공식 영문 이름이 무엇인지 알려주실 수 있나요?


> Entering new AgentExecutor chain...

Invoking: `find_player_id` with `{'player_name': 'Tarik Skubal'}`


[Tool Log] find_player_id 호출됨: Tarik Skubal
{'status': 'OK', 'id': '669373', 'fullName': 'Tarik Skubal', 'position': 'P', 'team': '', 'isPitcher': True, 'isTwoWay': False}
Invoking: `get_player_stats` with `{'player_id': '669373', 'time_info': '2025', 'stat_type': 'pitching'}`


[Tool Log] get_player_stats 호출됨: 669373, 2025, pitching
{'player_id': '669373', 'time': '2025', 'type': 'season', 'group': 'pitching', 'seasons': [{'season': '2025', 'type': 'season', 'group': 'pitching', 'raw': {'id': 669373, 'first_name': 'Tarik', 'last_name': 'Skubal', 'active': True, 'current_team': 'Detroit T